In [5]:
# ============================================================
# V3 — FEATURE NUMERICAL INTEGRITY AUDIT
# ============================================================

print("=" * 70)
print("V3 — NUMERICAL INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Count NaN values
# ------------------------------------------------------------

nan_counts = df_v3[V3_FEATURES].isna().sum()

nan_features = (
    nan_counts[nan_counts > 0]
    .sort_values(ascending=False)
)

print("\nFeatures containing NaN:")
print(f"  Number of affected features: {len(nan_features)}")

display(
    nan_features.to_frame("NaN_count")
)

# ------------------------------------------------------------
# 2. Count infinite values
# ------------------------------------------------------------

inf_counts = np.isinf(
    df_v3[V3_FEATURES].to_numpy()
).sum(axis=0)

inf_series = pd.Series(
    inf_counts,
    index=V3_FEATURES
)

inf_features = (
    inf_series[inf_series > 0]
    .sort_values(ascending=False)
)

print("\n" + "=" * 70)
print("INFINITE VALUES")
print("=" * 70)

if len(inf_features) == 0:
    print("✓ No +inf or -inf values found")
else:
    print("⚠ Infinite values found:")
    display(
        inf_features.to_frame("inf_count")
    )

# ------------------------------------------------------------
# 3. Overall numerical summary
# ------------------------------------------------------------

total_nan = int(
    df_v3[V3_FEATURES].isna().sum().sum()
)

total_inf = int(
    np.isinf(df_v3[V3_FEATURES].to_numpy()).sum()
)

total_values = (
    df_v3.shape[0] * len(V3_FEATURES)
)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Total feature values : {total_values:,}")
print(f"Total NaN values     : {total_nan:,}")
print(f"Total infinite      : {total_inf:,}")

print(
    f"NaN percentage       : "
    f"{100 * total_nan / total_values:.3f}%"
)

# ------------------------------------------------------------
# 4. Final decision
# ------------------------------------------------------------

assert total_inf == 0

print("\n✓ No infinite values")
print("✓ NaNs are allowed and will be handled by the model pipeline")

print("\n" + "=" * 70)
print("✓ V3 NUMERICAL INTEGRITY CHECK PASSED")
print("=" * 70)

V3 — NUMERICAL INTEGRITY AUDIT

Features containing NaN:
  Number of affected features: 71


,NaN_count
RHOB_ROLLSTD_3,188373
RHOB_ROLLMEAN_3,188373
RHOB_GRAD,188081
RHOB,187984
RHOB_ROLLMEAN_5,187981
...,...
GR_ROLLMEAN_9,1216
GR_ROLLMEAN_15,1151
GR_ROLLSTD_15,1151
GR_ROLLSTD_21,1091



INFINITE VALUES
✓ No +inf or -inf values found

SUMMARY
Total feature values : 108,754,024
Total NaN values     : 4,740,356
Total infinite      : 0
NaN percentage       : 4.359%

✓ No infinite values
✓ NaNs are allowed and will be handled by the model pipeline

✓ V3 NUMERICAL INTEGRITY CHECK PASSED


In [6]:
# ============================================================
# 10 — THREE-CLASS SPECIALIST
# CELL 2 — LOAD AND VERIFY FROZEN SPLITS
# ============================================================

print("=" * 70)
print("CELL 2 — LOAD FROZEN DEVELOPMENT SPLITS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load official FORCE-2020 split
# ------------------------------------------------------------

official_split = pd.read_csv(
    "../data/splits/force2020_split.csv"
)

print("\nOfficial split:")
print(
    official_split["SPLIT"]
    .value_counts()
)

# ------------------------------------------------------------
# 2. Extract official well groups
# ------------------------------------------------------------

official_train_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "TRAIN",
        "WELL"
    ]
)

public_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "PUBLIC",
        "WELL"
    ]
)

blind_test_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "BLIND_TEST",
        "WELL"
    ]
)

assert len(official_train_wells) == 98
assert len(public_wells) == 10
assert len(blind_test_wells) == 10

print("\nOfficial wells:")
print(f"  TRAIN      : {len(official_train_wells)}")
print(f"  PUBLIC     : {len(public_wells)}")
print(f"  BLIND_TEST : {len(blind_test_wells)}")

# ------------------------------------------------------------
# 3. Load frozen 78/20 development split
# ------------------------------------------------------------

dev_split = pd.read_csv(
    "../data/splits/development_split.csv"
)

print("\nDevelopment split columns:")
print(dev_split.columns.tolist())

assert {"WELL", "DEV_SPLIT"}.issubset(
    dev_split.columns
)

dev_train_wells = set(
    dev_split.loc[
        dev_split["DEV_SPLIT"] == "DEV_TRAIN",
        "WELL"
    ]
)

dev_val_wells = set(
    dev_split.loc[
        dev_split["DEV_SPLIT"] == "DEV_VALIDATION",
        "WELL"
    ]
)

assert len(dev_train_wells) == 78
assert len(dev_val_wells) == 20

assert dev_train_wells.isdisjoint(
    dev_val_wells
)

print("\nDevelopment wells:")
print(f"  DEV_TRAIN      : {len(dev_train_wells)}")
print(f"  DEV_VALIDATION : {len(dev_val_wells)}")

# ------------------------------------------------------------
# 4. Verify development wells belong only to official TRAIN
# ------------------------------------------------------------

assert dev_train_wells.issubset(
    official_train_wells
)

assert dev_val_wells.issubset(
    official_train_wells
)

# ------------------------------------------------------------
# 5. Verify no PUBLIC / BLIND contamination
# ------------------------------------------------------------

assert dev_train_wells.isdisjoint(
    public_wells
)

assert dev_val_wells.isdisjoint(
    public_wells
)

assert dev_train_wells.isdisjoint(
    blind_test_wells
)

assert dev_val_wells.isdisjoint(
    blind_test_wells
)

print("\n✓ Development wells are completely isolated")
print("✓ PUBLIC excluded")
print("✓ BLIND_TEST excluded")

# ------------------------------------------------------------
# 6. Assign development split to V3 rows
# ------------------------------------------------------------

# Remove any accidental split columns if present.
split_cols = [
    col for col in df_v3.columns
    if col in ["SPLIT", "DEV_SPLIT", "OFFICIAL_SPLIT"]
]

if split_cols:
    df_v3 = df_v3.drop(columns=split_cols)

# Keep only official TRAIN wells
df_dev_pool = df_v3[
    df_v3["WELL"].isin(official_train_wells)
].copy()

assert df_dev_pool["WELL"].nunique() == 98

# Merge frozen 78/20 split
df_dev_pool = df_dev_pool.merge(
    dev_split[["WELL", "DEV_SPLIT"]],
    on="WELL",
    how="left",
    validate="many_to_one",
)

assert df_dev_pool["DEV_SPLIT"].notna().all()

print("\nDevelopment pool:")
print(f"  Rows  : {len(df_dev_pool):,}")
print(f"  Wells : {df_dev_pool['WELL'].nunique()}")

# ------------------------------------------------------------
# 7. Create DEV_TRAIN and DEV_VALIDATION
# ------------------------------------------------------------

df_dev_train = df_dev_pool[
    df_dev_pool["DEV_SPLIT"] == "DEV_TRAIN"
].copy()

df_dev_val = df_dev_pool[
    df_dev_pool["DEV_SPLIT"] == "DEV_VALIDATION"
].copy()

print("\nDevelopment datasets:")
print(
    f"  DEV_TRAIN      : "
    f"{len(df_dev_train):,} rows / "
    f"{df_dev_train['WELL'].nunique()} wells"
)

print(
    f"  DEV_VALIDATION : "
    f"{len(df_dev_val):,} rows / "
    f"{df_dev_val['WELL'].nunique()} wells"
)

assert df_dev_train["WELL"].nunique() == 78
assert df_dev_val["WELL"].nunique() == 20

print("\n" + "=" * 70)
print("✓ FROZEN DEVELOPMENT SPLIT LOADED")
print("=" * 70)

CELL 2 — LOAD FROZEN DEVELOPMENT SPLITS

Official split:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

Official wells:
  TRAIN      : 98
  PUBLIC     : 10
  BLIND_TEST : 10

Development split columns:
['WELL', 'DEV_SPLIT', 'OFFICIAL_SPLIT']

Development wells:
  DEV_TRAIN      : 78
  DEV_VALIDATION : 20

✓ Development wells are completely isolated
✓ PUBLIC excluded
✓ BLIND_TEST excluded

Development pool:
  Rows  : 1,171,557
  Wells : 98

Development datasets:
  DEV_TRAIN      : 894,965 rows / 78 wells
  DEV_VALIDATION : 276,592 rows / 20 wells

✓ FROZEN DEVELOPMENT SPLIT LOADED


In [7]:
# ============================================================
# CELL 3 — BUILD 3-CLASS SPECIALIST DATASET + WEIGHTS
# ============================================================

print("=" * 70)
print("CELL 3 — 3-CLASS SPECIALIST DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define specialist classes
# ------------------------------------------------------------

SPECIALIST_CLASSES = np.array([
    30000.0,   # Sandstone
    65000.0,   # Shale
    65030.0,   # Sandstone/Shale
])

print("\nSpecialist classes:")
print("  30000 = Sandstone")
print("  65000 = Shale")
print("  65030 = Sandstone/Shale")

# ------------------------------------------------------------
# 2. Filter DEV_TRAIN
# ------------------------------------------------------------

df_spec_train = df_dev_train[
    df_dev_train[TARGET_COL].isin(SPECIALIST_CLASSES)
].copy()

# ------------------------------------------------------------
# 3. Filter DEV_VALIDATION
# ------------------------------------------------------------

df_spec_val = df_dev_val[
    df_dev_val[TARGET_COL].isin(SPECIALIST_CLASSES)
].copy()

# ------------------------------------------------------------
# 4. Verify class coverage
# ------------------------------------------------------------

train_classes = set(
    df_spec_train[TARGET_COL].unique()
)

val_classes = set(
    df_spec_val[TARGET_COL].unique()
)

assert train_classes == set(SPECIALIST_CLASSES)
assert val_classes == set(SPECIALIST_CLASSES)

# ------------------------------------------------------------
# 5. Verify well separation
# ------------------------------------------------------------

spec_train_wells = set(
    df_spec_train["WELL"].unique()
)

spec_val_wells = set(
    df_spec_val["WELL"].unique()
)

assert spec_train_wells.isdisjoint(
    spec_val_wells
)

print("\nSpecialist datasets:")
print(
    f"  DEV_TRAIN      : "
    f"{len(df_spec_train):,} rows / "
    f"{len(spec_train_wells)} wells"
)

print(
    f"  DEV_VALIDATION : "
    f"{len(df_spec_val):,} rows / "
    f"{len(spec_val_wells)} wells"
)

# ------------------------------------------------------------
# 6. Class distribution
# ------------------------------------------------------------

train_counts = (
    df_spec_train[TARGET_COL]
    .value_counts()
    .reindex(SPECIALIST_CLASSES)
)

val_counts = (
    df_spec_val[TARGET_COL]
    .value_counts()
    .reindex(SPECIALIST_CLASSES)
)

distribution = pd.DataFrame({
    "DEV_TRAIN": train_counts,
    "DEV_VALIDATION": val_counts,
})

distribution["TRAIN_%"] = (
    distribution["DEV_TRAIN"]
    / distribution["DEV_TRAIN"].sum()
    * 100
)

distribution["VAL_%"] = (
    distribution["DEV_VALIDATION"]
    / distribution["DEV_VALIDATION"].sum()
    * 100
)

print("\n" + "=" * 70)
print("SPECIALIST CLASS DISTRIBUTION")
print("=" * 70)

display(distribution)

# ------------------------------------------------------------
# 7. Build specialist ML matrices
# ------------------------------------------------------------

X_spec_train = df_spec_train[V3_FEATURES]
y_spec_train = df_spec_train[TARGET_COL]

X_spec_val = df_spec_val[V3_FEATURES]
y_spec_val = df_spec_val[TARGET_COL]

assert X_spec_train.shape[1] == 76
assert X_spec_val.shape[1] == 76

print("\nML matrices:")
print(f"  X_spec_train : {X_spec_train.shape}")
print(f"  y_spec_train : {y_spec_train.shape}")
print(f"  X_spec_val   : {X_spec_val.shape}")
print(f"  y_spec_val   : {y_spec_val.shape}")

# ------------------------------------------------------------
# 8. Calculate specialist-specific sqrt-balanced weights
# ------------------------------------------------------------

N_SPEC = len(y_spec_train)
K_SPEC = len(SPECIALIST_CLASSES)

specialist_class_weights = np.sqrt(
    N_SPEC /
    (K_SPEC * train_counts)
)

print("\n" + "=" * 70)
print("SPECIALIST SQRT-BALANCED CLASS WEIGHTS")
print("=" * 70)

for cls in SPECIALIST_CLASSES:
    print(
        f"{int(cls):6d} : "
        f"count={int(train_counts[cls]):8,d}  "
        f"weight={specialist_class_weights[cls]:.6f}"
    )

# ------------------------------------------------------------
# 9. Convert class weights to sample weights
# ------------------------------------------------------------

specialist_sample_weights = (
    y_spec_train
    .map(specialist_class_weights)
    .to_numpy(dtype=np.float64)
)

assert len(specialist_sample_weights) == len(
    y_spec_train
)

assert np.isfinite(
    specialist_sample_weights
).all()

assert (
    specialist_sample_weights > 0
).all()

print("\n✓ Every specialist training sample has a valid weight")

# ------------------------------------------------------------
# 10. Weight statistics
# ------------------------------------------------------------

print("\nSample-weight statistics:")
print(
    f"  Minimum : "
    f"{specialist_sample_weights.min():.6f}"
)

print(
    f"  Maximum : "
    f"{specialist_sample_weights.max():.6f}"
)

print(
    f"  Mean    : "
    f"{specialist_sample_weights.mean():.6f}"
)

print(
    f"  Median  : "
    f"{np.median(specialist_sample_weights):.6f}"
)

# ------------------------------------------------------------
# 11. Final checks
# ------------------------------------------------------------

assert len(X_spec_train) == len(
    specialist_sample_weights
)

assert set(
    y_spec_train.unique()
) == set(SPECIALIST_CLASSES)

assert set(
    y_spec_val.unique()
) == set(SPECIALIST_CLASSES)

print("\n" + "=" * 70)
print("✓ 3-CLASS SPECIALIST DATASET FROZEN")
print("✓ SPECIALIST TRAINING WEIGHTS FROZEN")
print("=" * 70)

CELL 3 — 3-CLASS SPECIALIST DATASET

Specialist classes:
  30000 = Sandstone
  65000 = Shale
  65030 = Sandstone/Shale

Specialist datasets:
  DEV_TRAIN      : 797,542 rows / 78 wells
  DEV_VALIDATION : 243,653 rows / 20 wells

SPECIALIST CLASS DISTRIBUTION


,DEV_TRAIN,DEV_VALIDATION,TRAIN_%,VAL_%
FORCE_2020_LITHOFACIES_LITHOLOGY,,,,
30000.0,131392,38013,16.474618,15.601285
65000.0,544884,176301,68.320414,72.357410
65030.0,121266,29339,15.204967,12.041305



ML matrices:
  X_spec_train : (797542, 76)
  y_spec_train : (797542,)
  X_spec_val   : (243653, 76)
  y_spec_val   : (243653,)

SPECIALIST SQRT-BALANCED CLASS WEIGHTS
 30000 : count= 131,392  weight=1.422433
 65000 : count= 544,884  weight=0.698496
 65030 : count= 121,266  weight=1.480630

✓ Every specialist training sample has a valid weight

Sample-weight statistics:
  Minimum : 0.698496
  Maximum : 1.480630
  Mean    : 0.936685
  Median  : 0.698496

✓ 3-CLASS SPECIALIST DATASET FROZEN
✓ SPECIALIST TRAINING WEIGHTS FROZEN


In [8]:
# ============================================================
# CELL 4 — 3-CLASS XGBOOST SPECIALIST
# ============================================================

import time

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

print("=" * 70)
print("CELL 4 — 3-CLASS XGBOOST SPECIALIST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Encode specialist labels
# ------------------------------------------------------------

specialist_classes = np.array([
    30000.0,
    65000.0,
    65030.0,
])

specialist_class_to_idx = {
    cls: idx
    for idx, cls in enumerate(specialist_classes)
}

y_spec_train_idx = (
    y_spec_train
    .map(specialist_class_to_idx)
    .to_numpy()
)

y_spec_val_idx = (
    y_spec_val
    .map(specialist_class_to_idx)
    .to_numpy()
)

assert set(np.unique(y_spec_train_idx)) == {0, 1, 2}
assert set(np.unique(y_spec_val_idx)) == {0, 1, 2}

print("\nEncoded classes:")
for cls, idx in specialist_class_to_idx.items():
    print(f"  {idx} → {int(cls)}")

# ------------------------------------------------------------
# 2. Create XGBoost model
# ------------------------------------------------------------

specialist_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

print("\nModel:")
print(specialist_xgb)

# ------------------------------------------------------------
# 3. Train
# ------------------------------------------------------------

print("\nTraining specialist...")

start_time = time.perf_counter()

specialist_xgb.fit(
    X_spec_train,
    y_spec_train_idx,
    sample_weight=specialist_sample_weights,
)

train_time = time.perf_counter() - start_time

print(
    f"✓ Training completed in "
    f"{train_time:.2f} seconds"
)

# ------------------------------------------------------------
# 4. Predict
# ------------------------------------------------------------

print("\nPredicting DEV_VALIDATION...")

start_time = time.perf_counter()

y_spec_pred_idx = specialist_xgb.predict(
    X_spec_val
)

prediction_time = time.perf_counter() - start_time

# Convert indices back to lithology codes
y_spec_pred = specialist_classes[
    y_spec_pred_idx.astype(int)
]

print(
    f"✓ Prediction completed in "
    f"{prediction_time:.2f} seconds"
)

# ------------------------------------------------------------
# 5. Overall metrics
# ------------------------------------------------------------

spec_accuracy = accuracy_score(
    y_spec_val,
    y_spec_pred
)

spec_balanced_accuracy = balanced_accuracy_score(
    y_spec_val,
    y_spec_pred
)

spec_macro_f1 = f1_score(
    y_spec_val,
    y_spec_pred,
    average="macro",
    zero_division=0,
)

spec_weighted_f1 = f1_score(
    y_spec_val,
    y_spec_pred,
    average="weighted",
    zero_division=0,
)

print("\n" + "=" * 70)
print("3-CLASS XGBOOST SPECIALIST RESULTS")
print("=" * 70)

print(
    f"\nAccuracy           : "
    f"{spec_accuracy:.6f}"
)

print(
    f"Balanced Accuracy  : "
    f"{spec_balanced_accuracy:.6f}"
)

print(
    f"Macro F1           : "
    f"{spec_macro_f1:.6f}"
)

print(
    f"Weighted F1        : "
    f"{spec_weighted_f1:.6f}"
)

# ------------------------------------------------------------
# 6. Per-class classification report
# ------------------------------------------------------------

specialist_report = classification_report(
    y_spec_val,
    y_spec_pred,
    labels=specialist_classes,
    target_names=[
        "30000 Sandstone",
        "65000 Shale",
        "65030 Sandstone/Shale",
    ],
    zero_division=0,
    output_dict=True,
)

specialist_report_df = pd.DataFrame(
    specialist_report
).T

print("\n" + "=" * 70)
print("SPECIALIST PER-CLASS REPORT")
print("=" * 70)

display(specialist_report_df)

# ------------------------------------------------------------
# 7. Save experiment summary in memory
# ------------------------------------------------------------

specialist_xgb_results = {
    "experiment": "3_class_XGBoost_specialist",
    "features": 76,
    "classes": 3,
    "n_estimators": 300,
    "max_depth": 3,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "accuracy": spec_accuracy,
    "balanced_accuracy": spec_balanced_accuracy,
    "macro_f1": spec_macro_f1,
    "weighted_f1": spec_weighted_f1,
    "train_time_sec": train_time,
    "prediction_time_sec": prediction_time,
}

print("\n" + "=" * 70)
print("✓ 3-CLASS XGBOOST SPECIALIST COMPLETE")
print("=" * 70)

CELL 4 — 3-CLASS XGBOOST SPECIALIST

Encoded classes:
  0 → 30000
  1 → 65000
  2 → 65030

Model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1, num_class=3, ...)

Training specialist...
✓ Training completed in 54.49 seconds

Predicting DEV_VALIDATION...
✓ Prediction completed in 0.78 seconds

3-CLASS XGBOOST SPECIAL

,precision,recall,f1-score,support
30000 Sandstone,0.713524,0.760450,0.736240,38013.000000
65000 Shale,0.885142,0.885208,0.885175,176301.000000
65030 Sandstone/Shale,0.339260,0.310201,0.324081,29339.000000
accuracy,0.796506,0.796506,0.796506,0.796506
macro avg,0.645976,0.651953,0.648499,243653.000000
weighted avg,0.792636,0.796506,0.794376,243653.000000



✓ 3-CLASS XGBOOST SPECIALIST COMPLETE


| Class | Precision | Recall | F1 |
|---|---:|---:|---:|
| 30000 Sandstone | .7135 | **.7605** | **.7362** |
| 65000 Shale | .8851 | **.8852** | **.8852** |
| 65030 Sandstone/Shale | .3393 | **.3102** | **.3241** |
| **Macro F1** | | | **.6485** |

In [9]:
# ============================================================
# CELL 5 — 3-CLASS SPECIALIST CONFUSION ANALYSIS
# ============================================================

from sklearn.metrics import confusion_matrix

print("=" * 70)
print("CELL 5 — 3-CLASS SPECIALIST CONFUSION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Raw confusion matrix
# ------------------------------------------------------------

cm_spec = confusion_matrix(
    y_spec_val,
    y_spec_pred,
    labels=SPECIALIST_CLASSES
)

cm_spec_df = pd.DataFrame(
    cm_spec,
    index=[
        "30000 Sandstone",
        "65000 Shale",
        "65030 Sandstone/Shale",
    ],
    columns=[
        "30000 Sandstone",
        "65000 Shale",
        "65030 Sandstone/Shale",
    ],
)

print("\n" + "=" * 70)
print("RAW CONFUSION MATRIX")
print("=" * 70)

display(cm_spec_df)

# ------------------------------------------------------------
# 2. Row-normalized confusion matrix
# ------------------------------------------------------------

cm_spec_norm = (
    cm_spec /
    cm_spec.sum(axis=1, keepdims=True)
)

cm_spec_norm_df = pd.DataFrame(
    cm_spec_norm,
    index=[
        "30000 Sandstone",
        "65000 Shale",
        "65030 Sandstone/Shale",
    ],
    columns=[
        "30000 Sandstone",
        "65000 Shale",
        "65030 Sandstone/Shale",
    ],
)

print("\n" + "=" * 70)
print("ROW-NORMALIZED CONFUSION MATRIX")
print("(Each row = true lithology)")
print("=" * 70)

display(
    cm_spec_norm_df.round(4)
)

# ------------------------------------------------------------
# 3. Detailed confusion percentages
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KEY CONFUSION PATHS")
print("=" * 70)

for i, true_cls in enumerate(SPECIALIST_CLASSES):

    total = cm_spec[i].sum()

    print(
        f"\nTrue {int(true_cls)} "
        f"({['Sandstone', 'Shale', 'Sandstone/Shale'][i]}):"
    )

    for j, pred_cls in enumerate(SPECIALIST_CLASSES):

        count = cm_spec[i, j]
        pct = 100 * count / total

        print(
            f"  → {int(pred_cls):5d}: "
            f"{count:7,d} samples "
            f"({pct:6.2f}%)"
        )

# ------------------------------------------------------------
# 4. Focus specifically on 65030
# ------------------------------------------------------------

idx_65030 = list(SPECIALIST_CLASSES).index(65030.0)

total_65030 = cm_spec[idx_65030].sum()

correct_65030 = cm_spec[
    idx_65030,
    idx_65030
]

to_sandstone = cm_spec[
    idx_65030,
    0
]

to_shale = cm_spec[
    idx_65030,
    1
]

print("\n" + "=" * 70)
print("65030 ERROR BREAKDOWN")
print("=" * 70)

print(
    f"\nTrue 65030 samples : {total_65030:,}"
)

print(
    f"Correct 65030      : "
    f"{correct_65030:,} "
    f"({100 * correct_65030 / total_65030:.2f}%)"
)

print(
    f"→ 30000 Sandstone   : "
    f"{to_sandstone:,} "
    f"({100 * to_sandstone / total_65030:.2f}%)"
)

print(
    f"→ 65000 Shale       : "
    f"{to_shale:,} "
    f"({100 * to_shale / total_65030:.2f}%)"
)

print("\n" + "=" * 70)
print("✓ SPECIALIST CONFUSION ANALYSIS COMPLETE")
print("=" * 70)

CELL 5 — 3-CLASS SPECIALIST CONFUSION ANALYSIS

RAW CONFUSION MATRIX


,30000 Sandstone,65000 Shale,65030 Sandstone/Shale
30000 Sandstone,28907,6346,2760
65000 Shale,5273,156063,14965
65030 Sandstone/Shale,6333,13905,9101



ROW-NORMALIZED CONFUSION MATRIX
(Each row = true lithology)


,30000 Sandstone,65000 Shale,65030 Sandstone/Shale
30000 Sandstone,0.7605,0.1669,0.0726
65000 Shale,0.0299,0.8852,0.0849
65030 Sandstone/Shale,0.2159,0.4739,0.3102



KEY CONFUSION PATHS

True 30000 (Sandstone):
  → 30000:  28,907 samples ( 76.05%)
  → 65000:   6,346 samples ( 16.69%)
  → 65030:   2,760 samples (  7.26%)

True 65000 (Shale):
  → 30000:   5,273 samples (  2.99%)
  → 65000: 156,063 samples ( 88.52%)
  → 65030:  14,965 samples (  8.49%)

True 65030 (Sandstone/Shale):
  → 30000:   6,333 samples ( 21.59%)
  → 65000:  13,905 samples ( 47.39%)
  → 65030:   9,101 samples ( 31.02%)

65030 ERROR BREAKDOWN

True 65030 samples : 29,339
Correct 65030      : 9,101 (31.02%)
→ 30000 Sandstone   : 6,333 (21.59%)
→ 65000 Shale       : 13,905 (47.39%)

✓ SPECIALIST CONFUSION ANALYSIS COMPLETE


In [11]:
# ============================================================
# CELL 6 — 65030 PROBABILITY THRESHOLD OPTIMIZATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

print("=" * 70)
print("CELL 6 — 65030 PROBABILITY THRESHOLD OPTIMIZATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Get specialist probabilities
# ------------------------------------------------------------

specialist_proba = specialist_xgb.predict_proba(
    X_spec_val
)

assert specialist_proba.shape == (
    len(X_spec_val),
    3
)

print("\nProbability matrix:")
print(f"  Shape: {specialist_proba.shape}")

# Probability columns:
# 0 -> 30000
# 1 -> 65000
# 2 -> 65030

P_30000 = specialist_proba[:, 0]
P_65000 = specialist_proba[:, 1]
P_65030 = specialist_proba[:, 2]

# ------------------------------------------------------------
# 2. Baseline argmax prediction
# ------------------------------------------------------------

baseline_idx = np.argmax(
    specialist_proba,
    axis=1
)

baseline_pred = specialist_classes[
    baseline_idx
]

baseline_macro_f1 = f1_score(
    y_spec_val,
    baseline_pred,
    average="macro",
    zero_division=0
)

baseline_65030_f1 = f1_score(
    y_spec_val,
    baseline_pred,
    labels=[65030.0],
    average="macro",
    zero_division=0
)

print("\nBaseline argmax:")
print(f"  Macro F1     : {baseline_macro_f1:.6f}")
print(f"  65030 F1     : {baseline_65030_f1:.6f}")

# ------------------------------------------------------------
# 3. Threshold sweep
#
# Rule:
#
# If P(65030) >= threshold:
#       predict 65030
#
# Otherwise:
#       choose between 30000 and 65000
#       using whichever has higher probability.
# ------------------------------------------------------------

thresholds = np.arange(
    0.20,
    0.61,
    0.01
)

results = []

for threshold in thresholds:

    pred_idx = np.where(
        P_65030 >= threshold,
        2,
        np.where(
            P_30000 >= P_65000,
            0,
            1
        )
    )

    pred = specialist_classes[
        pred_idx
    ]

    acc = accuracy_score(
        y_spec_val,
        pred
    )

    bal_acc = balanced_accuracy_score(
        y_spec_val,
        pred
    )

    macro = f1_score(
        y_spec_val,
        pred,
        average="macro",
        zero_division=0
    )

    f1_65030 = f1_score(
        y_spec_val,
        pred,
        labels=[65030.0],
        average="macro",
        zero_division=0
    )

    precision_65030 = precision_score(
        y_spec_val,
        pred,
        labels=[65030.0],
        average="macro",
        zero_division=0
    )

    recall_65030 = recall_score(
        y_spec_val,
        pred,
        labels=[65030.0],
        average="macro",
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro,
        "65030_precision": precision_65030,
        "65030_recall": recall_65030,
        "65030_f1": f1_65030,
    })

threshold_results = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Best threshold for 65030 F1
# ------------------------------------------------------------

best_65030 = threshold_results.loc[
    threshold_results["65030_f1"].idxmax()
]

# ------------------------------------------------------------
# 5. Best threshold for Macro F1
# ------------------------------------------------------------

best_macro = threshold_results.loc[
    threshold_results["macro_f1"].idxmax()
]

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST THRESHOLD FOR 65030 F1")
print("=" * 70)

print(
    f"\nThreshold       : "
    f"{best_65030['threshold']:.2f}"
)

print(
    f"65030 Precision : "
    f"{best_65030['65030_precision']:.6f}"
)

print(
    f"65030 Recall    : "
    f"{best_65030['65030_recall']:.6f}"
)

print(
    f"65030 F1        : "
    f"{best_65030['65030_f1']:.6f}"
)

print(
    f"Macro F1        : "
    f"{best_65030['macro_f1']:.6f}"
)

print(
    f"Accuracy        : "
    f"{best_65030['accuracy']:.6f}"
)

print("\n" + "=" * 70)
print("BEST THRESHOLD FOR MACRO F1")
print("=" * 70)

print(
    f"\nThreshold       : "
    f"{best_macro['threshold']:.2f}"
)

print(
    f"65030 Precision : "
    f"{best_macro['65030_precision']:.6f}"
)

print(
    f"65030 Recall    : "
    f"{best_macro['65030_recall']:.6f}"
)

print(
    f"65030 F1        : "
    f"{best_macro['65030_f1']:.6f}"
)

print(
    f"Macro F1        : "
    f"{best_macro['macro_f1']:.6f}"
)

print(
    f"Accuracy        : "
    f"{best_macro['accuracy']:.6f}"
)

# ------------------------------------------------------------
# 7. Show top thresholds
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP THRESHOLDS BY 65030 F1")
print("=" * 70)

display(
    threshold_results
    .sort_values("65030_f1", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP THRESHOLDS BY MACRO F1")
print("=" * 70)

display(
    threshold_results
    .sort_values("macro_f1", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("✓ 65030 THRESHOLD SWEEP COMPLETE")
print("=" * 70)

CELL 6 — 65030 PROBABILITY THRESHOLD OPTIMIZATION

Probability matrix:
  Shape: (243653, 3)

Baseline argmax:
  Macro F1     : 0.648499
  65030 F1     : 0.324081

BEST THRESHOLD FOR 65030 F1

Threshold       : 0.26
65030 Precision : 0.269960
65030 Recall    : 0.616108
65030 F1        : 0.375422
Macro F1        : 0.638977
Accuracy        : 0.718862

BEST THRESHOLD FOR MACRO F1

Threshold       : 0.35
65030 Precision : 0.306355
65030 Recall    : 0.448584
65030 F1        : 0.364071
Macro F1        : 0.652901
Accuracy        : 0.767871

TOP THRESHOLDS BY 65030 F1


,threshold,accuracy,balanced_accuracy,macro_f1,65030_precision,65030_recall,65030_f1
0,0.26,0.718862,0.674807,0.638977,0.269960,0.616108,0.375422
1,0.28,0.731996,0.675039,0.644217,0.278022,0.577423,0.375328
2,0.29,0.738181,0.674902,0.646467,0.282266,0.558778,0.375067
3,0.27,0.725442,0.674653,0.641457,0.273661,0.595760,0.375046
4,0.25,0.711606,0.674397,0.635818,0.265920,0.635502,0.374947
5,0.24,0.703968,0.673912,0.632193,0.262178,0.656396,0.374695
6,0.30,0.743824,0.674336,0.648223,0.286272,0.540032,0.374187
7,0.23,0.695588,0.672726,0.627909,0.258090,0.677187,0.373740
8,0.31,0.749381,0.673908,0.649999,0.290657,0.522547,0.373540
9,0.32,0.754598,0.673044,0.651266,0.295103,0.504278,0.372323



TOP THRESHOLDS BY MACRO F1


,threshold,accuracy,balanced_accuracy,macro_f1,65030_precision,65030_recall,65030_f1
0,0.35,0.767871,0.668478,0.652901,0.306355,0.448584,0.364071
1,0.36,0.771688,0.666681,0.652860,0.310189,0.431235,0.360831
2,0.34,0.763750,0.669996,0.652535,0.302437,0.466103,0.366843
3,0.37,0.775180,0.664540,0.652333,0.313394,0.412863,0.356317
4,0.33,0.759215,0.671315,0.651797,0.298498,0.484236,0.369330
5,0.38,0.778410,0.662292,0.651482,0.316829,0.395208,0.351705
6,0.32,0.754598,0.673044,0.651266,0.295103,0.504278,0.372323
7,0.39,0.781821,0.660224,0.650811,0.320754,0.377757,0.346929
8,0.31,0.749381,0.673908,0.649999,0.290657,0.522547,0.373540
9,0.40,0.784817,0.657934,0.649680,0.324600,0.360612,0.341660



✓ 65030 THRESHOLD SWEEP COMPLETE


In [13]:
# ============================================================
# CELL 7 — FIND TRAINED 12-CLASS MODELS
# ============================================================

print("=" * 70)
print("CELL 7 — SEARCHING FOR TRAINED 12-CLASS MODELS")
print("=" * 70)

candidate_models = []

# Take a snapshot BEFORE iterating
global_items = list(globals().items())

for name, obj in global_items:

    if name.startswith("_"):
        continue

    if hasattr(obj, "predict_proba") and hasattr(obj, "classes_"):

        try:
            classes = np.asarray(obj.classes_)

            if len(classes) == 12:

                candidate_models.append(
                    (
                        name,
                        type(obj).__name__,
                        classes
                    )
                )

        except Exception:
            continue


if len(candidate_models) == 0:

    print("\nNo trained 12-class model found.")

else:

    print(
        f"\nFound {len(candidate_models)} "
        f"trained 12-class model(s):\n"
    )

    for name, model_type, classes in candidate_models:

        print(f"Variable : {name}")
        print(f"Model    : {model_type}")
        print(f"Classes  : {classes}")
        print()


print("=" * 70)
print("✓ MODEL AUDIT COMPLETE")
print("=" * 70)

CELL 7 — SEARCHING FOR TRAINED 12-CLASS MODELS

No trained 12-class model found.
✓ MODEL AUDIT COMPLETE


In [14]:
# ============================================================
# CELL 8 — REBUILD V3 12-CLASS LIGHTGBM BASELINE
# ============================================================

import time

from lightgbm import LGBMClassifier

print("=" * 70)
print("CELL 8 — REBUILD V3 12-CLASS LIGHTGBM BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Full 12-class training labels
# ------------------------------------------------------------

FULL_CLASSES = np.array([
    30000.0,
    65000.0,
    65030.0,
    70000.0,
    70032.0,
    74000.0,
    80000.0,
    86000.0,
    88000.0,
    90000.0,
    93000.0,
    99000.0,
])

# ------------------------------------------------------------
# 2. Use the already-frozen V3 matrices
# ------------------------------------------------------------

X_full_train = df_dev_train[V3_FEATURES]
y_full_train = df_dev_train[TARGET_COL]

X_full_val = df_dev_val[V3_FEATURES]
y_full_val = df_dev_val[TARGET_COL]

assert X_full_train.shape == (894965, 76)
assert X_full_val.shape == (276592, 76)

# ------------------------------------------------------------
# 3. Recalculate V3 full-model sqrt-balanced weights
#    ONLY from DEV_TRAIN
# ------------------------------------------------------------

full_counts = (
    y_full_train
    .value_counts()
    .reindex(FULL_CLASSES)
)

assert full_counts.notna().all()

N_FULL = len(y_full_train)
K_FULL = len(FULL_CLASSES)

full_class_weights = np.sqrt(
    N_FULL /
    (K_FULL * full_counts)
)

full_sample_weights = (
    y_full_train
    .map(full_class_weights)
    .to_numpy(dtype=np.float64)
)

assert np.isfinite(full_sample_weights).all()
assert (full_sample_weights > 0).all()

print("\nFull-model class weights:")

for cls in FULL_CLASSES:
    print(
        f"{int(cls):6d} : "
        f"count={int(full_counts[cls]):8,d}  "
        f"weight={full_class_weights[cls]:.6f}"
    )

# ------------------------------------------------------------
# 4. Create exactly the previous V3 LightGBM model
# ------------------------------------------------------------

full_lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=12,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    n_jobs=-1,
    random_state=42,
    verbosity=-1,
)

print("\nModel:")
print(full_lgbm)

# ------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------

print("\nTraining full 12-class baseline...")

start_time = time.perf_counter()

full_lgbm.fit(
    X_full_train,
    y_full_train,
    sample_weight=full_sample_weights,
)

full_train_time = time.perf_counter() - start_time

print(
    f"✓ Training completed in "
    f"{full_train_time:.2f} seconds"
)

# ------------------------------------------------------------
# 6. Predict labels
# ------------------------------------------------------------

print("\nPredicting DEV_VALIDATION...")

start_time = time.perf_counter()

full_pred = full_lgbm.predict(
    X_full_val
)

full_prediction_time = time.perf_counter() - start_time

print(
    f"✓ Prediction completed in "
    f"{full_prediction_time:.2f} seconds"
)

# ------------------------------------------------------------
# 7. Obtain probabilities
# ------------------------------------------------------------

full_proba = full_lgbm.predict_proba(
    X_full_val
)

assert full_proba.shape == (
    len(X_full_val),
    12
)

# Verify class ordering
print("\nModel class ordering:")
print(full_lgbm.classes_)

assert np.array_equal(
    full_lgbm.classes_,
    FULL_CLASSES
)

# ------------------------------------------------------------
# 8. Baseline metrics
# ------------------------------------------------------------

full_accuracy = accuracy_score(
    y_full_val,
    full_pred
)

full_balanced_accuracy = balanced_accuracy_score(
    y_full_val,
    full_pred
)

full_macro_f1 = f1_score(
    y_full_val,
    full_pred,
    average="macro",
    zero_division=0
)

full_weighted_f1 = f1_score(
    y_full_val,
    full_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("REBUILT V3 LIGHTGBM BASELINE")
print("=" * 70)

print(f"\nAccuracy           : {full_accuracy:.6f}")
print(f"Balanced Accuracy  : {full_balanced_accuracy:.6f}")
print(f"Macro F1           : {full_macro_f1:.6f}")
print(f"Weighted F1        : {full_weighted_f1:.6f}")

print("\n" + "=" * 70)
print("✓ FULL 12-CLASS BASELINE REBUILT")
print("✓ PROBABILITIES AVAILABLE FOR HIERARCHICAL TEST")
print("=" * 70)

CELL 8 — REBUILD V3 12-CLASS LIGHTGBM BASELINE

Full-model class weights:
 30000 : count= 131,392  weight=0.753404
 65000 : count= 544,884  weight=0.369965
 65030 : count= 121,266  weight=0.784229
 70000 : count=  43,889  weight=1.303571
 70032 : count=   7,938  weight=3.065186
 74000 : count=   1,193  weight=7.906644
 80000 : count=  22,389  weight=1.825135
 86000 : count=     845  weight=9.394724
 88000 : count=   4,315  weight=4.157402
 90000 : count=   3,306  weight=4.749643
 93000 : count=     141  weight=22.998677
 99000 : count=  13,407  weight=2.358558

Model:
LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
               n_jobs=-1, num_class=12, objective='multiclass', random_state=42,
               reg_lambda=1, subsample=0.8, verbosity=-1)

Training full 12-class baseline...
✓ Training completed in 234.66 seconds

Predicting DEV_VALIDATION...
✓ Prediction completed in 25.77 seconds

Model class ordering:
[30000. 65000. 65030. 70000. 70032. 74000. 

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Metric | Previous V3 | Rebuilt |
|---|---:|---:|
| Accuracy | 0.713141 | **0.713141** |
| Balanced Accuracy | 0.407197 | **0.407197** |
| Macro F1 | 0.347298 | **0.347298** |
| Weighted F1 | 0.710178 | **0.710178** |

In [15]:
# ============================================================
# CELL 9 — HIERARCHICAL 12-CLASS + 3-CLASS SPECIALIST
# ============================================================

print("=" * 70)
print("CELL 9 — HIERARCHICAL SPECIALIST INTEGRATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Baseline full-model predictions
# ------------------------------------------------------------

hier_pred = full_pred.copy()

# ------------------------------------------------------------
# 2. Identify samples predicted as one of the specialist
#    classes by the full 12-class model
# ------------------------------------------------------------

specialist_mask = np.isin(
    full_pred,
    SPECIALIST_CLASSES
)

n_specialist_candidates = specialist_mask.sum()

print("\nFull-model predictions:")
print(
    f"  Total validation samples : "
    f"{len(full_pred):,}"
)

print(
    f"  Specialist candidates    : "
    f"{n_specialist_candidates:,}"
)

print(
    f"  Candidate percentage      : "
    f"{100 * n_specialist_candidates / len(full_pred):.2f}%"
)

# ------------------------------------------------------------
# 3. Extract specialist features
# ------------------------------------------------------------

X_hier_specialist = X_full_val.loc[
    specialist_mask
]

assert len(X_hier_specialist) == (
    n_specialist_candidates
)

# ------------------------------------------------------------
# 4. Specialist probabilities
# ------------------------------------------------------------

hier_spec_proba = specialist_xgb.predict_proba(
    X_hier_specialist
)

assert hier_spec_proba.shape == (
    n_specialist_candidates,
    3
)

P_spec_30000 = hier_spec_proba[:, 0]
P_spec_65000 = hier_spec_proba[:, 1]
P_spec_65030 = hier_spec_proba[:, 2]

# ------------------------------------------------------------
# 5. Apply specialist threshold
# ------------------------------------------------------------

SPECIALIST_THRESHOLD = 0.35

specialist_hier_idx = np.where(
    P_spec_65030 >= SPECIALIST_THRESHOLD,
    2,
    np.where(
        P_spec_30000 >= P_spec_65000,
        0,
        1
    )
)

specialist_hier_pred = SPECIALIST_CLASSES[
    specialist_hier_idx
]

# ------------------------------------------------------------
# 6. Replace only specialist-group predictions
# ------------------------------------------------------------

hier_pred[specialist_mask] = (
    specialist_hier_pred
)

# ------------------------------------------------------------
# 7. Count how many predictions changed
# ------------------------------------------------------------

changed_mask = (
    hier_pred != full_pred
)

n_changed = changed_mask.sum()

print("\nHierarchical changes:")
print(
    f"  Specialist candidates : "
    f"{n_specialist_candidates:,}"
)

print(
    f"  Predictions changed   : "
    f"{n_changed:,}"
)

print(
    f"  Change rate           : "
    f"{100 * n_changed / len(full_pred):.2f}%"
)

# ------------------------------------------------------------
# 8. Evaluate hierarchical model
# ------------------------------------------------------------

hier_accuracy = accuracy_score(
    y_full_val,
    hier_pred
)

hier_balanced_accuracy = balanced_accuracy_score(
    y_full_val,
    hier_pred
)

hier_macro_f1 = f1_score(
    y_full_val,
    hier_pred,
    average="macro",
    zero_division=0
)

hier_weighted_f1 = f1_score(
    y_full_val,
    hier_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("HIERARCHICAL MODEL RESULTS")
print("=" * 70)

print(
    f"\nAccuracy           : "
    f"{hier_accuracy:.6f}"
)

print(
    f"Balanced Accuracy  : "
    f"{hier_balanced_accuracy:.6f}"
)

print(
    f"Macro F1           : "
    f"{hier_macro_f1:.6f}"
)

print(
    f"Weighted F1        : "
    f"{hier_weighted_f1:.6f}"
)

# ------------------------------------------------------------
# 9. Compare against full LightGBM baseline
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASELINE vs HIERARCHICAL")
print("=" * 70)

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Weighted F1",
    ],
    "V3 LightGBM": [
        full_accuracy,
        full_balanced_accuracy,
        full_macro_f1,
        full_weighted_f1,
    ],
    "Hierarchical": [
        hier_accuracy,
        hier_balanced_accuracy,
        hier_macro_f1,
        hier_weighted_f1,
    ],
})

comparison["Change"] = (
    comparison["Hierarchical"]
    - comparison["V3 LightGBM"]
)

display(
    comparison.style.format({
        "V3 LightGBM": "{:.6f}",
        "Hierarchical": "{:.6f}",
        "Change": "{:+.6f}",
    })
)

# ------------------------------------------------------------
# 10. 65030-specific result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("65030 PERFORMANCE")
print("=" * 70)

baseline_65030_f1_full = f1_score(
    y_full_val,
    full_pred,
    labels=[65030.0],
    average="macro",
    zero_division=0,
)

hier_65030_f1 = f1_score(
    y_full_val,
    hier_pred,
    labels=[65030.0],
    average="macro",
    zero_division=0,
)

baseline_65030_recall = recall_score(
    y_full_val,
    full_pred,
    labels=[65030.0],
    average="macro",
    zero_division=0,
)

hier_65030_recall = recall_score(
    y_full_val,
    hier_pred,
    labels=[65030.0],
    average="macro",
    zero_division=0,
)

print(
    f"\nBaseline 65030 F1     : "
    f"{baseline_65030_f1_full:.6f}"
)

print(
    f"Hierarchical 65030 F1 : "
    f"{hier_65030_f1:.6f}"
)

print(
    f"Change                : "
    f"{hier_65030_f1 - baseline_65030_f1_full:+.6f}"
)

print(
    f"\nBaseline 65030 Recall     : "
    f"{baseline_65030_recall:.6f}"
)

print(
    f"Hierarchical 65030 Recall : "
    f"{hier_65030_recall:.6f}"
)

print(
    f"Change                   : "
    f"{hier_65030_recall - baseline_65030_recall:+.6f}"
)

print("\n" + "=" * 70)
print("✓ HIERARCHICAL EXPERIMENT COMPLETE")
print("=" * 70)

CELL 9 — HIERARCHICAL SPECIALIST INTEGRATION

Full-model predictions:
  Total validation samples : 276,592
  Specialist candidates    : 244,871
  Candidate percentage      : 88.53%

Hierarchical changes:
  Specialist candidates : 244,871
  Predictions changed   : 25,863
  Change rate           : 9.35%


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



HIERARCHICAL MODEL RESULTS

Accuracy           : 0.694669
Balanced Accuracy  : 0.406418
Macro F1           : 0.345889
Weighted F1        : 0.700754

BASELINE vs HIERARCHICAL


,Metric,V3 LightGBM,Hierarchical,Change
0,Accuracy,0.713141,0.694669,-0.018471
1,Balanced Accuracy,0.407197,0.406418,-0.000779
2,Macro F1,0.347298,0.345889,-0.001408
3,Weighted F1,0.710178,0.700754,-0.009423



65030 PERFORMANCE

Baseline 65030 F1     : 0.345220
Hierarchical 65030 F1 : 0.351727
Change                : +0.006507

Baseline 65030 Recall     : 0.348274
Hierarchical 65030 Recall : 0.432019
Change                   : +0.083745

✓ HIERARCHICAL EXPERIMENT COMPLETE


| Metric | Baseline |
|---|---:|
| Accuracy | **0.713141** |
| Balanced Accuracy | **0.407197** |
| Macro F1 | **0.347298** |
| Weighted F1 | **0.710178** |

-------------------------------


| Metric | Hierarchical | Change |
|---|---:|---:|
| Accuracy | 0.694669 | **−0.018471** |
| Balanced Accuracy | 0.406418 | −0.000779 |
| Macro F1 | 0.345889 | **−0.001408** |
| Weighted F1 | 0.700754 | −0.009423 |

In [16]:
# ============================================================
# CELL 10 — SPECIALIST GATING STRATEGY COMPARISON
# ============================================================

print("=" * 70)
print("CELL 10 — SPECIALIST GATING STRATEGY COMPARISON")
print("=" * 70)

THRESHOLD = 0.35

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def evaluate_prediction_strategy(name, prediction):
    
    accuracy = accuracy_score(
        y_full_val,
        prediction
    )
    
    balanced = balanced_accuracy_score(
        y_full_val,
        prediction
    )
    
    macro = f1_score(
        y_full_val,
        prediction,
        average="macro",
        zero_division=0
    )
    
    weighted = f1_score(
        y_full_val,
        prediction,
        average="weighted",
        zero_division=0
    )
    
    f1_65030 = f1_score(
        y_full_val,
        prediction,
        labels=[65030.0],
        average="macro",
        zero_division=0
    )
    
    recall_65030 = recall_score(
        y_full_val,
        prediction,
        labels=[65030.0],
        average="macro",
        zero_division=0
    )
    
    return {
        "strategy": name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "macro_f1": macro,
        "weighted_f1": weighted,
        "65030_f1": f1_65030,
        "65030_recall": recall_65030,
    }


# ------------------------------------------------------------
# Strategy A
# Full prediction belongs to specialist group
# ------------------------------------------------------------

mask_all_three = np.isin(
    full_pred,
    SPECIALIST_CLASSES
)

X_gate_A = X_full_val.loc[
    mask_all_three
]

proba_gate_A = specialist_xgb.predict_proba(
    X_gate_A
)

P_A_30000 = proba_gate_A[:, 0]
P_A_65000 = proba_gate_A[:, 1]
P_A_65030 = proba_gate_A[:, 2]

pred_A_idx = np.where(
    P_A_65030 >= THRESHOLD,
    2,
    np.where(
        P_A_30000 >= P_A_65000,
        0,
        1
    )
)

pred_A_specialist = SPECIALIST_CLASSES[
    pred_A_idx
]

pred_A = full_pred.copy()

pred_A[mask_all_three] = pred_A_specialist


# ------------------------------------------------------------
# Strategy B
# Specialist invoked ONLY when full model predicts 65030
# ------------------------------------------------------------

mask_65030 = (
    full_pred == 65030.0
)

X_gate_B = X_full_val.loc[
    mask_65030
]

proba_gate_B = specialist_xgb.predict_proba(
    X_gate_B
)

P_B_30000 = proba_gate_B[:, 0]
P_B_65000 = proba_gate_B[:, 1]
P_B_65030 = proba_gate_B[:, 2]

pred_B_idx = np.where(
    P_B_65030 >= THRESHOLD,
    2,
    np.where(
        P_B_30000 >= P_B_65000,
        0,
        1
    )
)

pred_B_specialist = SPECIALIST_CLASSES[
    pred_B_idx
]

pred_B = full_pred.copy()

pred_B[mask_65030] = pred_B_specialist


# ------------------------------------------------------------
# Evaluate all strategies
# ------------------------------------------------------------

baseline_result = evaluate_prediction_strategy(
    "V3 LightGBM baseline",
    full_pred
)

strategy_A_result = evaluate_prediction_strategy(
    "A: specialist on 30000/65000/65030",
    pred_A
)

strategy_B_result = evaluate_prediction_strategy(
    "B: specialist only on 65030",
    pred_B
)

gating_results = pd.DataFrame([
    baseline_result,
    strategy_A_result,
    strategy_B_result,
])

print("\n" + "=" * 70)
print("GATING STRATEGY RESULTS")
print("=" * 70)

display(
    gating_results.style.format({
        "accuracy": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "macro_f1": "{:.6f}",
        "weighted_f1": "{:.6f}",
        "65030_f1": "{:.6f}",
        "65030_recall": "{:.6f}",
    })
)

# ------------------------------------------------------------
# Number of overrides
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERRIDE COUNTS")
print("=" * 70)

print(
    f"\nStrategy A candidates : "
    f"{mask_all_three.sum():,}"
)

print(
    f"Strategy A changes    : "
    f"{np.sum(pred_A != full_pred):,}"
)

print(
    f"\nStrategy B candidates : "
    f"{mask_65030.sum():,}"
)

print(
    f"Strategy B changes    : "
    f"{np.sum(pred_B != full_pred):,}"
)

print("\n" + "=" * 70)
print("✓ GATING COMPARISON COMPLETE")
print("=" * 70)

CELL 10 — SPECIALIST GATING STRATEGY COMPARISON


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



GATING STRATEGY RESULTS


,strategy,accuracy,balanced_accuracy,macro_f1,weighted_f1,65030_f1,65030_recall
0,V3 LightGBM baseline,0.713141,0.407197,0.347298,0.710178,0.345220,0.348274
1,A: specialist on 30000/65000/65030,0.694669,0.406418,0.345889,0.700754,0.351727,0.432019
2,B: specialist only on 65030,0.716669,0.405168,0.346024,0.710509,0.327646,0.309281



OVERRIDE COUNTS

Strategy A candidates : 244,871
Strategy A changes    : 25,863

Strategy B candidates : 29,858
Strategy B changes    : 3,808

✓ GATING COMPARISON COMPLETE


| Strategy | Accuracy | Balanced Acc. | Macro F1 | 65030 F1 | 65030 Recall |
|---|---:|---:|---:|---:|---:|
| **V3 LightGBM baseline** | **0.713141** | **0.407197** | **0.347298** | **0.345220** | **0.348274** |
| A: specialist on 3 classes | 0.694669 | 0.406418 | 0.345889 | **0.351727** | **0.432019** |
| B: specialist only on 65030 | 0.716669 | 0.405168 | 0.346024 | 0.327646 | 0.309281 |



The standalone 3-class specialist is good at distinguishing the three classes when it is forced to choose among them, but that does not mean its decisions are better than the 12-class model's decisions in the full geological context.

| Model | Feature setup | Macro F1 |
|---|---|---:|
| Older gradient XGBoost | V2 + gradients | **0.369926** |
| Older gradient LightGBM | V2 + gradients | 0.347732 |
| V3 LightGBM | finer rolling + gradients | 0.347298 |
| V3 XGBoost | finer rolling + gradients | 0.339586 |
| V3 ExtraTrees | finer rolling + gradients | 0.359845 |
| 3-class specialist | specialist only | 0.648499 |
| Specialist threshold | specialist only | **0.652901** |